# Bài 4: Tô loang và Thành phần liên thông (Flood Fill and Connected Components)

Khi đã có một ảnh nhị phân (ví dụ từ phân ngưỡng ở Bài 3), câu hỏi tự nhiên tiếp theo là: *có bao nhiêu đốm (blob) tách rời nhau, và chúng nằm ở đâu?* Có hai công cụ trả lời câu hỏi này:

- **Tô loang (flood fill)** phát triển một vùng từ một điểm ảnh hạt giống (seed) duy nhất, lan sang tất cả các điểm ảnh lân cận liên thông có giá trị tương tự.
- **Gán nhãn thành phần liên thông (connected component labeling)** tìm *tất cả* các vùng như vậy trong ảnh nhị phân cùng lúc, gán cho mỗi vùng một mã định danh (ID) duy nhất.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['fruit.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tạo một ảnh nhị phân gồm nhiều đốm

Hãy đọc một ảnh xám chụp vài loại trái cây trên nền tối, dùng kiến thức Bài 3 để phân ngưỡng ảnh và loại bỏ nhiễu muối.

In [ ]:
img = cv2.imread('../img/fruit.jpg', cv2.IMREAD_GRAYSCALE)
_, im_bin = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
kernel = np.ones((3, 3), np.uint8)
im_bin2 = cv2.morphologyEx(im_bin, cv2.MORPH_OPEN, kernel)

fig, axes = plt.subplots(1, 3, figsize=(10,4))
axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(im_bin, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Thresholded')
axes[1].axis('off')
axes[2].imshow(im_bin2, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('After opening')
axes[2].axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: Stan Birchfield</p>

## Tô loang từ một điểm hạt giống

Tô loang bắt đầu từ một điểm ảnh hạt giống và lan ra tất cả các điểm ảnh liên thông có giá trị nằm trong một ngưỡng dung sai so với giá trị hạt giống, tô chúng bằng một màu mới. Thuật toán cổ điển duy trì một "biên (frontier)" gồm các điểm ảnh còn cần thăm (cài đặt bằng ngăn xếp (stack) hoặc hàng đợi (queue)): lấy ra một điểm ảnh; nếu nó thuộc tiền cảnh và chưa được tô, đánh dấu đã tô và đẩy 4 điểm lân cận liên thông của nó vào ngăn xếp. Lặp lại cho đến khi ngăn xếp rỗng, lúc đó mọi điểm ảnh có thể đi tới từ hạt giống qua một đường gồm các điểm ảnh tiền cảnh đều đã được tìm thấy.

In [ ]:
def flood_fill_stack(binary, seed):
    """Classic flood fill: an explicit stack holds the frontier of pixels still to visit."""
    filled = np.zeros_like(binary, dtype=bool)
    h, w = binary.shape
    stack = [seed]
    while stack:
        x, y = stack.pop()
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        if filled[y, x] or binary[y, x] == 0:
            continue
        filled[y, x] = True
        stack.extend([(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)])  # 4-connected neighbors
    return filled

Ở đây ta khởi tạo thuật toán với một điểm ảnh bên trong quả chuối đầu tiên.

In [ ]:
seed = (60, 110)  # (x, y) inside the first banana
im_region_filled = flood_fill_stack(im_bin2, seed)

im_filled = cv2.cvtColor(im_bin2, cv2.COLOR_GRAY2RGB)
im_filled[im_region_filled] = (255, 140, 0)

plt.imshow(im_filled)
plt.scatter(*seed, c='red', s=30, marker='x')
plt.title(f'Flood fill from scratch ({im_region_filled.sum()} pixels filled)')
plt.axis('off')
plt.show()

Dưới đây là ảnh động minh họa thuật toán dựa trên ngăn xếp (trên ảnh đã giảm độ phân giải để dễ quan sát).

<img src="../animations/floodfill_demo.gif" alt="Ảnh động minh họa tô loang dựa trên ngăn xếp, cho thấy vùng đã tô và biên các điểm ảnh còn trong ngăn xếp lan rộng dần từ hạt giống" style="display: block; margin: 0 auto; max-width: 500px;">

## Tô loang với OpenCV

`cv2.floodFill` làm đúng việc tương tự, chỉ là nhanh hơn đáng kể vì được biên dịch.

In [ ]:
# floodFill needs a mask 2 pixels larger than the image, and modifies the image in place
im_filled_cv2 = cv2.cvtColor(im_bin2, cv2.COLOR_GRAY2RGB)
im_mask = np.zeros(np.add(im_bin2.shape, (2, 2)), dtype=np.uint8)

cv2.floodFill(im_filled_cv2, im_mask, seed, (255, 140, 0))

plt.imshow(im_filled_cv2)
plt.scatter(*seed, c='red', s=30, marker='x')
plt.title('Flood fill from one seed (red x)')
plt.axis('off')
plt.show()

im_region_filled_cv2 = np.all(im_filled_cv2 == (255, 140, 0), axis=-1)
print(f'The two implementations match exactly: {np.array_equal(im_region_filled, im_region_filled_cv2)}')

## Thành phần liên thông: gán nhãn mọi đốm cùng lúc

Thay vì chọn hạt giống thủ công, thuật toán gán nhãn thành phần liên thông quét toàn bộ ảnh và gán cho mỗi đốm một nhãn số nguyên riêng. Một cách đơn giản để cài đặt: quét ảnh, và mỗi khi gặp một điểm ảnh tiền cảnh chưa có nhãn, chạy tô loang với một màu mới. Ảnh động này minh họa quá trình đó.

<img src="../animations/components_demo.gif" alt="Ảnh động minh họa thuật toán thành phần liên thông kiểu quét-và-tô-loang, gán cho mỗi đốm một màu mới khi phát hiện" style="display: block; margin: 0 auto; max-width: 500px;">

Thuật toán thành phần liên thông cổ điển (`cv2.connectedComponentsWithStats`) được gọi là *union-find*, và nó cần hai lượt duyệt qua ảnh, cùng với một lượt duyệt bảng tương đương. Chi tiết thuật toán được lược bỏ cho ngắn gọn, nhưng kết quả giống hệt thuật toán quét-và-tô-loang vừa nêu.

In [ ]:
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(im_bin2, connectivity=8)

# Give each label a distinct random color for visualization
colors = np.array([(0,0,0), (255,0,0), (0,200,0), (0,0,255), (180,180,0), (0,180,180), (180,0,180)])

im_colored = colors[labels].astype(np.uint8)

plt.imshow(im_colored)
for label in range(1, num_labels):
    cx, cy = centroids[label]
    plt.text(cx, cy, str(label), color='white', ha='center', va='center', fontsize=12, fontweight='bold')
plt.title('Connected components, colored and labeled')
plt.axis('off')
plt.show()

Dù chọn cách cài đặt nào, thuật toán gán nhãn thành phần liên thông cũng trả về các thống kê tiện dụng (ví dụ: hộp bao (bounding box), diện tích, trọng tâm). Thông tin bổ sung này gần như miễn phí, vì hầu như không tốn thêm phép tính nào (như chúng ta sẽ thấy ở Bài 5).

In [ ]:
print(f'Found {num_labels - 1} blobs (plus the background as label 0)\n')
print(f'{"label":>5} {"area":>6} {"centroid":>16}')
for label in range(1, num_labels):
    area = stats[label, cv2.CC_STAT_AREA]
    cx, cy = centroids[label]
    print(f'{label:>5} {area:>6} ({cx:6.1f}, {cy:6.1f})')

## Lọc đốm theo kích thước

Một ứng dụng phổ biến của thành phần liên thông là loại bỏ các đốm nhỏ giống nhiễu và chỉ giữ lại các đốm đáng kể. Ở đây ta chọn một ngưỡng loại bỏ các quả táo nhỏ (đốm 2 và 6).

In [ ]:
min_area = 3000
img_minsize = np.zeros_like(im_bin2)
for label in range(1, num_labels):
    if stats[label, cv2.CC_STAT_AREA] > min_area:
        img_minsize[labels == label] = 255

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(im_bin2, cmap='gray')
axes[0].set_title('All blobs')
axes[0].axis('off')
axes[1].imshow(img_minsize, cmap='gray')
axes[1].set_title(f'Blobs with area > {min_area}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Bài tập

1. Đổi `connectivity=8` thành `connectivity=4` trong `connectedComponentsWithStats`. Hãy dựng một ảnh nhị phân (ví dụ một bậc thang chéo gồm các điểm ảnh đơn lẻ) mà ở đó liên thông 4 và liên thông 8 cho ra số thành phần *khác nhau*.
2. Dùng `cv2.floodFill` với dung sai `loDiff`/`upDiff` khác 0 trên một ảnh xám (không phải nhị phân), và mô tả điều gì thay đổi.